# Canonical WOMD Paper Pipeline

Production entrypoint for the frozen WOMD paper protocol. The notebook handles authorized WOMD acquisition/build; `scripts/run_canonical_womd_pipeline.py` owns the Stage 1–7 gates so notebook cells cannot silently drift from the repository contract.


In [ ]:
from pathlib import Path
import shutil, subprocess, sys
from google.colab import auth, drive

auth.authenticate_user()
drive.mount('/content/drive')
ROOT = Path('/content/predictive-pc-fmcw')
DATA = Path('/content/womd')
PERSIST = Path('/content/drive/MyDrive/predictive_pc_fmcw_canonical')
DATA.mkdir(parents=True, exist_ok=True)
PERSIST.mkdir(parents=True, exist_ok=True)
REPO = 'https://github.com/panagiotagrosdouli/predictive-pc-fmcw-vehicular-communications..git'
if not ROOT.exists():
    subprocess.run(['git','clone','--depth','1',REPO,str(ROOT)], check=True)
else:
    subprocess.run(['git','-C',str(ROOT),'pull','--ff-only'], check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','-e',f'{ROOT}[ml,paper]','--no-build-isolation'], check=True)


## Build the frozen WOMD corpora

Paper recovery uses training shards 00000–00049 of 01000 and validation shards 00000–00039 of 00150 from WOMD v1.3.1, max 16 vehicles/scenario. Validation is exported only as `official_validation`.


In [ ]:
BUCKET = 'gs://waymo_open_dataset_motion_v_1_3_1/uncompressed/scenario'
train_dir = DATA/'training'; val_dir = DATA/'validation'
train_dir.mkdir(exist_ok=True); val_dir.mkdir(exist_ok=True)
def fetch_split(split_name, count, total, destination):
    for index in range(count):
        name = f'{split_name}.tfrecord-{index:05d}-of-{total:05d}'
        target = destination/name
        if not target.exists():
            subprocess.run(['gcloud','storage','cp',f'{BUCKET}/{split_name}/{name}',str(target)], check=True)
fetch_split('training', 50, 1000, train_dir)
fetch_split('validation', 40, 150, val_dir)
train_npz = DATA/'womd_training_paper.npz'
val_npz = DATA/'womd_validation_paper.npz'
subprocess.run([sys.executable,str(ROOT/'scripts/01_build_official_womd_samples.py'),*map(str,sorted(train_dir.glob('training.tfrecord-*'))),'--output',str(train_npz),'--max-vehicles','16'],cwd=ROOT,check=True)
subprocess.run([sys.executable,str(ROOT/'scripts/01_build_official_womd_samples.py'),*map(str,sorted(val_dir.glob('validation.tfrecord-*'))),'--output',str(val_npz),'--max-vehicles','16','--fixed-split','official_validation'],cwd=ROOT,check=True)


## Stage 1 — provenance and leakage gates

This hashes every selected TFRecord shard, audits both NPZ corpora, compares the historical training fingerprint without forcing old counts, and fail-closes on scenario leakage or validation-role violations.


In [ ]:
subprocess.run([sys.executable,str(ROOT/'scripts/run_canonical_womd_pipeline.py'),'--repo-root',str(ROOT),'--data-root',str(DATA),'--train-npz',str(train_npz),'--validation-npz',str(val_npz),'--mode','stage1'],check=True)
shutil.copytree(ROOT/'artifacts/paper_final/01_data',PERSIST/'paper_final/01_data',dirs_exist_ok=True)
print('Stage 1 PASS. Review historical_fingerprint.json before claiming exact historical reproduction.')


## Full canonical run — Stages 3–7

Stage 2 must already be frozen in the repository. The runner executes development baselines and lambda sweep, freezes the declared development-only lambda choice, trains 4 objectives × 5 seeds, requires 20 checkpoints, evaluates untouched validation, runs canonical paired scheduling, and produces scenario-level statistics.


In [ ]:
LAMBDA_LINK = 0.2
LAMBDA_OUTAGE = 0.1
RATIONALE = 'Selected from the declared development-only sweep; frozen before Stage 5.'
validation_pattern = str(val_dir/'validation.tfrecord-*')
subprocess.run([sys.executable,str(ROOT/'scripts/run_canonical_womd_pipeline.py'),'--repo-root',str(ROOT),'--data-root',str(DATA),'--train-npz',str(train_npz),'--validation-npz',str(val_npz),'--validation-glob',validation_pattern,'--lambda-link',str(LAMBDA_LINK),'--lambda-outage',str(LAMBDA_OUTAGE),'--selection-rationale',RATIONALE,'--mode','full'],check=True)
shutil.copytree(ROOT/'artifacts/paper_final',PERSIST/'paper_final',dirs_exist_ok=True)
print('Stages 1–7 complete; Stage 8 remains gated by release readiness.')
